# TV4 — Gradient-Boosted Trees (Spark MLlib)

Notebook thực hiện đúng phần TV4 – Phase 4: quét lưới thô trên 25% Train, huấn luyện lại tối đa 3 cấu hình trên toàn bộ Train, chọn số cây bằng `evaluateEachIteration` trên Validation, chọn threshold tối đa F1 trên Validation và permutation importance trên Validation.

**Không đọc hoặc sử dụng Test trong notebook này.** Dữ liệu đầu vào phải là `features_v1` đã đóng băng bởi TV2.

## 1. Môi trường Spark

Nhóm đã ghi trong `feature_spec.md`: Python 3.13 / PySpark 4.0.1 / Java 17. Colab có thể thay đổi phiên bản Python theo runtime; cell dưới ghim PySpark và Java, đồng thời in phiên bản thực tế để kiểm tra.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk-headless
%pip install -q "pyspark==4.0.1" "numpy>=1.26,<3" "pandas>=2.1,<3" "pyarrow>=14,<24" "scikit-learn>=1.4,<2" "matplotlib>=3.7,<4"

import os, sys
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!java -version 2>&1 | head -1
import pyspark
print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
assert pyspark.__version__ == "4.0.1", "Runtime chưa dùng đúng PySpark 4.0.1; hãy restart runtime và chạy lại từ đầu."


## 2. Kết nối Drive, repo và bảng đặc trưng

Drive cần có shortcut của thư mục `BigData_Topic20` trong `MyDrive`. Dữ liệu được copy về ổ cục bộ Colab để giảm thời gian đọc Parquet. Mô hình lưu ở `02_models/gbt`; kết quả TV4 lưu riêng trong `03_results`.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/BDA-Nhom_8')
if not (REPO / 'src' / 'model_gbt.py').is_file():
    subprocess.run(
        ['git', 'clone', '--branch', 'tv4/gbt',
         'https://github.com/BAC164/BDA-Nhom_8.git', str(REPO)],
        check=True,
    )
sys.path.insert(0, str(REPO))

DRIVE_ROOT = Path('/content/drive/MyDrive/BigData_Topic20')
FEATURES_ON_DRIVE = DRIVE_ROOT / '01_data_frozen' / 'features_v1'
MODEL_DIR = DRIVE_ROOT / '02_models' / 'gbt'
RESULTS_DIR = DRIVE_ROOT / '03_results'
FEATURES_DIR = Path('/content/features_v1')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not (FEATURES_ON_DRIVE / 'train').is_dir() or not (FEATURES_ON_DRIVE / 'val').is_dir():
    raise FileNotFoundError(f'Không tìm thấy train/val trong {FEATURES_ON_DRIVE}. Kiểm tra shortcut Drive và quyền truy cập của TV4.')
if not (FEATURES_DIR / 'train').is_dir():
    shutil.copytree(FEATURES_ON_DRIVE, FEATURES_DIR, dirs_exist_ok=True)

print('REPO:', REPO)
print('FEATURES_DIR:', FEATURES_DIR)
print('MODEL_DIR:', MODEL_DIR)
print('RESULTS_DIR:', RESULTS_DIR)


## 3. Khởi tạo Spark và đọc Train/Validation

Chỉ đọc hai split cần cho Phase 4. Không tạo lại split và không đọc `test/`.

In [ ]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('BDA_Nhom8_TV4_GBT')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '16')
    .config('spark.default.parallelism', '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

train = spark.read.parquet(str(FEATURES_DIR / 'train'))
val = spark.read.parquet(str(FEATURES_DIR / 'val'))
n_train, n_val = train.count(), val.count()
print('Train rows:', n_train, '| Validation rows:', n_val)
print('Train schema:')
train.printSchema()
assert n_train == 376749, f'Số dòng Train không đúng feature_spec.md: {n_train}'
assert n_val == 81606, f'Số dòng Validation không đúng feature_spec.md: {n_val}'
assert set(r[0] for r in train.select('Delay').distinct().collect()) == {0, 1}
assert set(r[0] for r in val.select('Delay').distinct().collect()) == {0, 1}
print('Không đọc Test; Train/Validation đúng schema và chứa cả hai lớp.')


## 4. Hai tầng tuning và lưu mô hình tốt nhất

Lưới thô theo phân công: `maxDepth ∈ {3, 5, 7}` × `stepSize ∈ {0.05, 0.1, 0.2}` × `subsamplingRate ∈ {0.7, 1.0}`; cố định `maxBins=300`, `seed=42`, `maxIter=100`. Tối đa 3 cấu hình đứng đầu theo ROC-AUC/PR-AUC Validation được huấn luyện lại trên toàn Train. `evaluateEachIteration` chọn số cây theo validation loss; nếu API không chạy ở runtime, module ghi cảnh báo và dùng fallback có ghi log.

In [ ]:
from src.model_gbt import train_gbt
from pprint import pprint

gbt_result = train_gbt(
    train_df=train,
    val_df=val,
    model_dir=MODEL_DIR,
    results_dir=RESULTS_DIR,
    sample_fraction=0.25,
    max_bins=300,
    max_iter=100,
    seed=42,
    top_k=3,
)
print('\nSelected configuration and validation results:')
pprint(gbt_result['config'])


## 5. Permutation importance trên Validation

Đây là phần giải thích mô hình ưu tiên theo phân công TV4. Mỗi nhóm đặc trưng được hoán vị cùng nhau; `Delay` và `row_id` không bị xáo trộn. Mức giảm ROC-AUC càng lớn thì nhóm đặc trưng càng quan trọng đối với dự đoán trên Validation. Không dùng `featureImportances` làm kết luận chính.

In [ ]:
from src.model_gbt import permutation_importance_val
from IPython.display import display

permutation_results = permutation_importance_val(
    model=gbt_result['model'],
    val_df=val,
    results_dir=RESULTS_DIR,
    repeats=3,
    seed=42,
)
display(permutation_results)


## 6. Kiểm tra log và biểu đồ kết quả

Các file được lưu trên Drive sau từng lần thử nghiệm để tránh mất log khi Colab ngắt phiên.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

experiments = pd.read_csv(RESULTS_DIR / 'gbt_experiments.csv')
display(experiments.sort_values(['stage', 'val_roc_auc'], ascending=[True, False]).tail(25))

fig, ax = plt.subplots(figsize=(9, 5))
importance_plot = permutation_results.sort_values('mean_auc_decrease', ascending=True)
ax.barh(importance_plot['feature_group'], importance_plot['mean_auc_decrease'],
        xerr=importance_plot['std_auc_decrease'].fillna(0))
ax.set_xlabel('Validation ROC-AUC decrease after permutation')
ax.set_title('TV4 — Grouped permutation importance (GBT)')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'gbt_permutation_importance.png', dpi=160)
plt.show()

print('Model:', gbt_result['model_path'])
print('Config:', gbt_result['config_path'])
print('Validation predictions:', gbt_result['predictions_path'])
print('Experiment log:', gbt_result['experiment_log'])
print('Permutation importance:', RESULTS_DIR / 'gbt_permutation_importance.csv')


## Kết thúc Phase 4 của TV4

Trước khi sang Phase 5, TV4 gửi TV5: `gbt_experiments.csv`, `predictions_val_gbt.parquet`, `models/gbt/best_model`, `models/gbt/config.json`, và bảng/biểu đồ permutation importance. **Chưa chạy test tại notebook này.** Việc chạy test chỉ thực hiện một lần sau khi cả nhóm chốt và gắn tag `final-models`.